Agents -> Create custom tools and also inbuilt tools that is connecting to the LLMs and getting the result. Here we are using three:
1. Wikipedia and Arxiv
2.langsmith

In [1]:
!pip install arxiv


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 2.3 MB/s eta 0:00:00a 0:00:01
  Attempting uninstall: requests
    Found existing installation: requests 2.34.2
    Uninstalling requests-2.34.2:
      Successfully uninstalled requests-2.34.2
  Attempting uninstall: lxml
    Found existing installation: lxml 5.3.0
    Uninstalling lxml-5.3.0:
      Successfully uninstalled lxml-5.3.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [arxiv]32m1/3 [lxml]


In [ ]:
# in bulit tool 
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper


In [13]:
api_wrapper= WikipediaAPIWrapper(top_k_results=1,doc_content_chars_max=200)
wiki=WikipediaQueryRun(api_wrapper=api_wrapper)

In [14]:
wiki.name

'wikipedia'

In [ ]:
#custom tool
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

#loading the data or files
loader= WebBaseLoader("https://docs.smith.langchain.com/")
docs=loader.load()
## splitting the context into smaller chunks
documents=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200).split_documents(docs)

#converting that chunks to the embeddings(number) and storing in FAISS
vectordb=FAISS.from_documents(documents, OpenAIEmbeddings())

#creating the retriever(interface that retrieve the data from the vectordb)
retriever= vectordb.as_retriever()
retriever

VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x10f4063c0>, search_kwargs={})

In [9]:
from langchain_core.tools import create_retriever_tool
retriever_tool= create_retriever_tool(retriever, "lang_smith_search", " Search for information about LangSmith. For any questions about LangSmith, you must use this tool!")

In [10]:
retriever_tool.name

'lang_smith_search'

In [12]:
## Arxiv Tool in built
from langchain_community.tools import ArxivQueryRun #This converts the arXiv search capability into a Tool that an Agent can use.
from langchain_community.utilities import ArxivAPIWrapper #api that have realted data

arxiv_wrapper=ArxivAPIWrapper(top_k_results=1, doc_content_chars_max=100)
arxiv=ArxivQueryRun(api_wrapper=arxiv_wrapper)
arxiv.name

'arxiv'

In [15]:
tools=[wiki, arxiv, retriever_tool]

In [16]:
tools

[WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(wiki_client=<module 'wikipedia' from '/Users/rajeshnalla/Documents/New_aiprojects/Langchain/venv/lib/python3.13/site-packages/wikipedia/__init__.py'>, top_k_results=1, lang='en', load_all_available_meta=False, doc_content_chars_max=200)),
 ArxivQueryRun(api_wrapper=ArxivAPIWrapper(arxiv_search=<class 'arxiv.Search'>, arxiv_exceptions=(<class 'arxiv.ArxivError'>, <class 'arxiv.UnexpectedEmptyPageError'>, <class 'arxiv.HTTPError'>), top_k_results=1, ARXIV_MAX_QUERY_LENGTH=300, continue_on_failure=False, load_max_docs=100, load_all_available_meta=False, doc_content_chars_max=100)),
 StructuredTool(name='lang_smith_search', description=' Search for information about LangSmith. For any questions about LangSmith, you must use this tool!', args_schema=<class 'langchain_core.tools.retriever.RetrieverInput'>, func=<function create_retriever_tool.<locals>.func at 0x11c5c8180>, coroutine=<function create_retriever_tool.<locals>.afunc at 0x11c5c84

In [17]:
from dotenv import load_dotenv
import os
load_dotenv()
os.environ['OPENAI_API_KEY']=os.getenv("OPENAI_API_KEY")
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-3.5-turbo-0125", temperature=0)

In [20]:
import langchain
print(langchain.__version__)

1.3.9


In [24]:
# from langchain import hub


# prompt=hub.pull("hwchase17/openai-functions-agent")
# prompt.messages
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    ("human", "{input}")
])

In [26]:
### Agents

from langchain.agents import create_openai_tools_agent
agent=create_openai_tools_agent(llm, tools, prompt)

ImportError: cannot import name 'create_openai_tools_agent' from 'langchain.agents' (/Users/rajeshnalla/Documents/New_aiprojects/Langchain/venv/lib/python3.13/site-packages/langchain/agents/__init__.py)